# Timing Analysis

## What you'll learn

- Load pipeline timing data with `PipelineTimings.from_delta()`
- Inspect step-level phase timings as a DataFrame
- Plot stacked bar charts of step phase breakdowns
- Drill into per-execution timings for individual steps
- Compare mean execution timings across steps
- Inspect subprocess launch durations and missing command evidence

**PipelineTimings** loads recorded phase durations from Delta Lake. Use its
DataFrames and plots to find slow steps and inspect their executions.

**Prerequisites:** [Your First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Exploring Results](../01-getting-started/02-exploring-results.ipynb).  
**Estimated time:** 10 minutes  
**GPU required:** No.

In [ ]:
from __future__ import annotations

from artisan.operations.curator import Filter
from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.orchestration.runners import LocalRunner
from artisan.utils import tutorial_setup
from artisan.visualization import PipelineTimings

In [ ]:
from functools import partial
from tempfile import TemporaryDirectory

# Keep this tutorial's stores and source files in one owned temporary workspace.
workspace = TemporaryDirectory(prefix="artisan-tutorial-")
tutorial_setup = partial(tutorial_setup, base_dir=workspace.name)

In [ ]:
env = tutorial_setup("timing_analysis")

Build a multi-step pipeline to generate timing data. We use batching on
the DataTransformer step to create two execution units whose timings we can
compare.

In [ ]:
pipeline = PipelineManager.create(
    name="timing_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    definition=DataGenerator,
    name="generate",
    params={"count": 10, "seed": 42},
    step_runner=LocalRunner(),
)
pipeline.run(
    definition=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    batch_strategy={"artifacts_per_unit": 5},
    step_runner=LocalRunner(),
)
pipeline.run(
    definition=MetricCalculator,
    name="score",
    inputs={"dataset": output("transform", "dataset")},
    step_runner=LocalRunner(),
)
pipeline.run(
    definition=Filter,
    name="filter",
    inputs={"passthrough": output("transform", "dataset")},
    params={
        "criteria": [
            {"metric": "distribution.median", "operator": "gt", "value": 0.3},
        ]
    },
    step_runner=LocalRunner(),
)
pipeline.run(
    definition=DataTransformer,
    name="refine",
    inputs={"dataset": output("filter", "passthrough")},
    params={"seed": 99},
    step_runner=LocalRunner(),
)

pipeline.finalize()

## Load timing data

`PipelineTimings.from_delta()` reads timing metadata from steps and executions.
Run-scoped queries exclude cached source durations because that work happened
in an earlier run. Execution timing includes fresh successful executions from
succeeded or partial steps. Use `inspect_commands` for command evidence from
failed attempts.

In [ ]:
timings = PipelineTimings.from_delta(env.delta_root)

## Step-level timings

`.step_timings()` preserves actual current step durations and any recorded phase
observations. Unavailable phases remain null; reused source timings are not
charged to the current step. The plot uses the available observations without
replacing missing values with zeros.

In [ ]:
timings.step_timings()

## Plot step phase breakdown

The `.plot_steps()` method renders a stacked horizontal bar chart showing
how each step's time breaks down by phase. Compare the longest bars and
the phases contributing to them.

In [ ]:
timings.plot_steps()

You can filter to specific steps with the `step_numbers` parameter --
useful for focusing on a subset of a long pipeline.

In [ ]:
timings.plot_steps(step_numbers=[0, 1, 2])

## Execution-level timings

Inspect individual execution units within a slow step. Each unit processes a
batch of artifacts; units can run in parallel.

Step phases measure orchestration, such as input resolution, batching, and
commit. Execution phases measure the operation's setup, execution, and recording.

In [ ]:
timings.execution_timings(step_number=1)

Each row is one execution unit. Step 1 used `artifacts_per_unit=5` with
10 inputs, so you should see 2 execution units.

## Subprocess launch timings

Inspect subprocess launch evidence separately from execution phases:


In [ ]:
timings.command_timings(step_number=1)

These Python examples launch no subprocesses, so they have a complete empty
recording and no `launch_seconds`. A command operation records time spent
creating its subprocess. That duration does not measure tool readiness, queueing,
or time waiting for output, and it is already included in the execution phases.

A missing duration can also mean evidence was unavailable or a launch failed.
Check `recording_status` before treating a null value as “no subprocess.” Use
`inspect_commands` for the recorded command details; see
[Inspect Pipeline Results](../../how-to-guides/inspecting-provenance.md).

## Execution statistics

For steps with many execution units, summary statistics are more useful
than raw timings. The `.execution_stats()` method computes mean, standard
deviation, min, and max for each phase across all execution units in a
step. Step 1 has two units: enough to illustrate the calculation, but too few
to draw broad performance conclusions.

In [ ]:
timings.execution_stats(step_number=1)

## Compare execution timings across steps

`.plot_execution_stats()` compares mean execution-phase durations across steps.
Use it to identify operations that take longer per execution unit.

In [ ]:
timings.plot_execution_stats()

Choose batching and concurrency settings using representative workloads; these
two execution units do not establish a performance benchmark.

## Next steps

- [Batching and Performance](../04-batching/01-batching-and-performance.ipynb) -- Tune batching parameters based on timing insights
- [Provenance Graphs](01-provenance-graphs.ipynb) -- Visualize the pipeline you profiled
- [Execution Flow](../../concepts/execution-flow.md) -- Understand the phases that timing analysis measures
- [Configuring Execution](../../how-to-guides/configuring-execution.md) -- Resource and scheduling configuration